# Running Simulation Evaluation Locally

This tutorial shows how to run simulation evaluation locally on your own machine using Docker, and serves as a foundation for building an evaluation setup tailored to your own compute resources.

The simulation runs inside a **Docker container**, while the **policy server** runs on the host. The two communicate over **gRPC**. By the end of this notebook you will be able to:
- Run a single-task evaluation manually (policy server + Docker sim)
- Automate multi-task, multi-GPU evaluation with `run_evaluation.py`
- Load, aggregate, and visualize results using the built-in dashboard and plotting utilities

## Prerequisites

- NVIDIA GPU with NVIDIA Container Toolkit installed
- Docker installed
- The **`Python (vla_foundry)`** Jupyter kernel — install it once from the repo root:
  ```bash
  bash tutorials/install_kernel.sh
  ```
  This syncs dependencies (inference, dashboard, etc.) and registers the kernel.
- Restart your notebook kernel after installing the vla_foundry kernel and select it for execution.
- A trained VLA Foundry checkpoint (see Step 1 below)

## Overview

| Step | Objective | Key File(s) |
|------|-----------|-------------|
| 0 | Set configuration variables | *(this notebook)* |
| 1 | Model checkpoint (HF Hub or local) | `vla_foundry/hf_hub.py` |
| 2 | Run a single-task evaluation manually | `vla_foundry/inference/robotics/inference_policy.py`, Docker |
| 3 | Automate evaluation: multi-task, multi-GPU, parameter comparison | `vla_foundry/eval/run_evaluation.py` |
| 4 | View and analyze results | `vla_foundry/eval/data_loading.py`, `vla_foundry/eval/stats.py`, `vla_foundry/eval/results_explorer.py` |
| 5 | Troubleshooting reference | *(this notebook)* |

## Architecture

When getting started, it is easiest to run evaluation in **two terminals**: one for the policy server (host) and one for the simulation (Docker). They communicate over gRPC on `localhost:50051`:

```
 Terminal 1 (Host)                         Terminal 2 (Docker)
┌─────────────────────────┐              ┌─────────────────────────┐
│  inference_policy.py    │    gRPC      │  Drake simulation       │
│  (policy server)        │◄────────────►│  (lbm_eval Docker)      │
│                         │              │                         │
│  Loads checkpoint,      │              │  Runs evaluation        │
│  generates actions      │              │  episodes, checks       │
│  from observations      │              │  success criteria       │
└─────────────────────────┘              └─────────────────────────┘
```

### Prerequisites
- NVIDIA GPU
- [NVIDIA Container Toolkit](https://docs.nvidia.com/datacenter/cloud-native/container-toolkit/latest/install-guide.html) installed
- Docker installed
- `uv` installed (see [main README](../README.md#installation))
- Project dependencies synced: `uv sync --group inference`
- Sufficient local disk space for rollout data

---

## Step 0: Configuration

Set the variables below once. Every subsequent cell references them, so you only need to change values here.

In [ ]:
import os
import subprocess

# Change to the repo root (works regardless of where the notebook is opened from)
repo_root = subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip()
os.chdir(repo_root)

In [ ]:
# ---- Configuration ----

TASK = "BimanualPutRedBellPepperInBin"  # Task name (PascalCase)
NUM_EPISODES = "0:2"  # Seed range (start:end, exclusive). Paper default: 0:200
OUTPUT_DIR = "tutorials/rollouts"  # Where all evaluation results are written
MANUAL_DIR = f"{OUTPUT_DIR}_manual"  # Where Step 2 manual eval results are written (separate layout)
NUM_GPUS = 3  # GPUs for automated evaluation
POLICY_PORT = 50051  # gRPC port for policy server
MAX_SAMPLE_SIZE = 200
# (preliminary statistical test);
# set to 200 to be consistent with technical report results.

DOCKER_IMAGE = "toyotaresearch/lbm-eval-oss:vla-foundry"

# ---- Model ----
# HuggingFace Hub model ID or local path to a trained checkpoint directory.
# The model is downloaded and cached automatically on first use.
# Browse available models at: https://huggingface.co/TRI-ML
CHECKPOINT_DIR = "TRI-ML/Foundry-Qwen3VLA-2B"

---

## Step 1: Model Checkpoint

`CHECKPOINT_DIR` above accepts three forms, resolved in this order:

1. **`hf://` or `s3://` URI** — used as-is (e.g. `hf://TRI-ML/Foundry-Qwen3VLA-2B`, `s3://bucket/my-exp`).
2. **Local path that exists on disk** (relative or absolute) — used as a local checkpoint directory. This is the common "train locally, eval locally" case.
3. **Bare HuggingFace repo ID** (e.g. `TRI-ML/Foundry-Qwen3VLA-2B`) — when none of the above match, the ID is treated as a HF model and downloaded/cached automatically on first use.

Because of rule 2, a relative local path like `experiments/my_run/` is picked up as local without needing a prefix — so long as the directory actually exists when the cell runs. If you mistype the path, you'll get a HF 404 instead of a file-not-found.

The checkpoint directory must contain:
- `config.yaml` — model architecture config
- `config_normalizer.yaml` — normalization statistics
- `checkpoints/` — at least one `checkpoint_*.pt` file (the latest is used by default)

---

## Step 2: Manual Single-Task Evaluation

This section walks through running a single task end-to-end so you can see how the pieces fit together. Results are written to a separate `tutorials/rollouts_manual/` directory — Step 3's `run_evaluation.py` writes to `OUTPUT_DIR` with the structured layout expected by the analysis tools in Step 4.

### 2a. Launch the Policy Server

The policy server loads the model checkpoint and waits for gRPC observation requests from the simulator.

| Parameter | Description | Default |
|-----------|-------------|---------|
| `--num_flow_steps` | Diffusion denoising steps | 8 |
| `--open_loop_steps` | Actions executed before re-planning | 8 |
| `--device` | `cuda` or `cpu` | `cuda` |

The next cell launches the server in the background and waits until it is accepting connections before returning.

**Optional:** To visualize observations and actions in real time, uncomment the `VISUALIZER` line below. This requires `uv sync --group visualization`. The server log (`tutorials/rollouts_manual/.policy_server.log`) will contain a URL like:

```
[rerun_backend] Open in browser: http://localhost:9090/?url=rerun%2Bhttp://127.0.0.1:9876/proxy
```

In [ ]:
import socket
import subprocess
import time
from pathlib import Path

# Launch policy server in the background.
# Logs are written to {MANUAL_DIR}/.policy_server.log.
manual_path = Path(MANUAL_DIR)
manual_path.mkdir(parents=True, exist_ok=True)
policy_log = manual_path / ".policy_server.log"
policy_log_fh = policy_log.open("w")  # noqa: SIM115 — must outlive Popen
policy_env = {**os.environ, "CUDA_VISIBLE_DEVICES": "0"}
# Uncomment to enable the rerun visualizer (requires: uv sync --group visualization):
# policy_env["VISUALIZER"] = "rerun"

uv_groups = ["--group", "inference"]
if "VISUALIZER" in policy_env:
    uv_groups += ["--group", "visualization"]

policy_proc = subprocess.Popen(
    [
        "uv",
        "run",
        *uv_groups,
        "python",
        "vla_foundry/inference/robotics/inference_policy.py",
        "--checkpoint_directory",
        CHECKPOINT_DIR,
        "--num_flow_steps",
        "8",
        "--open_loop_steps",
        "8",
        "--device",
        "cuda",
    ],
    env=policy_env,
    stdout=policy_log_fh,
    stderr=subprocess.STDOUT,
)

# Wait for the server to accept connections before proceeding (timeout 600s).
# Model loading typically takes 30-60s depending on GPU.
print(f"Policy server launched (PID {policy_proc.pid}), waiting for port {POLICY_PORT}...")
server_ready = False
for i in range(600):
    if policy_proc.poll() is not None:
        break
    try:
        with socket.create_connection(("localhost", POLICY_PORT), timeout=1):
            server_ready = True
            break
    except OSError:
        if i > 0 and i % 10 == 0:
            print(f"  Still waiting... ({i}s elapsed)")
        time.sleep(1)

# Always print the log so far.
policy_log_fh.flush()
print("\n--- Policy server log ---")
print(policy_log.read_text())

if server_ready:
    print(f"Policy server ready on localhost:{POLICY_PORT}")
elif policy_proc.poll() is not None:
    raise RuntimeError(f"Policy server exited with code {policy_proc.returncode}.")
else:
    raise TimeoutError(f"Policy server did not start on port {POLICY_PORT} within 600s.")

### 2b. Pull the Docker Image

The simulation runs inside a Docker container that includes the Drake physics engine and the LBM evaluation harness.

> ⚠️ **Heavy downloads ahead.** Pulling the `lbm-eval-oss` Docker image (~several GB) and the Foundry model checkpoint from HuggingFace takes time. Both are cached, so re-runs are fast. Under `jupyter nbconvert`, raise `--ExecutePreprocessor.timeout` (e.g. 3600) or pre-pull Docker and model in a terminal.


In [ ]:
!docker pull {DOCKER_IMAGE}

### 2c. Run Evaluation Episodes

The Docker container connects to the policy server over gRPC, runs each episode, and writes results to the mounted `tutorials/rollouts_manual/` directory.

**Key environment variables:**

| Variable | Description | Default |
|----------|-------------|---------|
| `LAUNCH_DEMONSTRATION_INDICES` | Episode seed range (`start:end`, exclusive) | `0:200` |
| `NUM_PROCESSES` | Parallel episodes inside the container | `1` |
| `POLICY_HOST` | gRPC policy server hostname | `localhost` |
| `POLICY_PORT` | gRPC policy server port | `50051` |
| `RECORD_VIDEO` | Save MP4 videos (`1` = yes) | `0` |
| `VIDEO_FPS` | Video frame rate | `10` |

In [ ]:
# Create output directory with open permissions (Docker runs as a different user).
!mkdir -p {MANUAL_DIR} && chmod 777 {MANUAL_DIR}

# Run evaluation episodes inside Docker.
!docker run --rm --network host \
    --runtime=nvidia \
    --gpus all \
    --device /dev/dri \
    --group-add video \
    --group-add $(stat -c '%g' /dev/dri/renderD128) \
    -e NVIDIA_DRIVER_CAPABILITIES=all \
    -e LAUNCH_DEMONSTRATION_INDICES={NUM_EPISODES} \
    -e RECORD_VIDEO=1 \
    -v $(pwd)/{MANUAL_DIR}:/tmp/lbm/rollouts \
    {DOCKER_IMAGE} \
    bash launch_sim.sh {TASK}

### 2d. Cleanup

Stop the background policy server and any running Docker evaluation containers.

In [ ]:
# Stop the policy server.
if policy_proc.poll() is None:
    policy_proc.terminate()
    policy_proc.wait(timeout=10)
    print("Policy server stopped.")
else:
    print(f"Policy server already exited (code {policy_proc.returncode}).")

# Kill any remaining policy servers on the port.
!kill $(lsof -ti :{POLICY_PORT}) 2>/dev/null \
    && echo "Killed process on port {POLICY_PORT}." || true

# Stop any Docker evaluation containers.
!docker kill $(docker ps -q --filter ancestor={DOCKER_IMAGE}) 2>/dev/null \
    && echo "Docker containers stopped." \
    || echo "No running containers found."

---

## Step 3: Automated Multi-Task Evaluation

[`run_evaluation.py`](../vla_foundry/eval/run_evaluation.py) automates Steps 2a–2c above: it launches policy servers (one per GPU), distributes tasks across GPUs with no idle time, and prints progress. When a GPU finishes a task it immediately picks up the next one.

The cell below compares two models side-by-side by running each with a different `--model_name`. Results appear as separate entries in the dashboard.

### Task selection

If `--tasks` is omitted, `run_evaluation.py` runs the paper's **19 default tasks** (16 main + 3 unseen, matching `rename.yaml`). To run a subset, pass specific task names to `--tasks`.

### Setting `--max_sample_size`

Every `run_evaluation.py` invocation requires `--max_sample_size`. This is the **maximum number of episodes per model per task** you plan to ever collect, and it configures the stopping boundary for the sequential statistical test used to compare models. It must be decided **before seeing any results** to maintain statistical validity — changing it after the fact invalidates the test.

Choose a value based on your compute budget. The paper uses 200 episodes, so `--max_sample_size 200` is a good default. `--num_episodes` can be smaller — you can start with a subset (e.g., `0:5` to verify the pipeline works) and collect more later up to your budget. See [`STATISTICAL_COMPARISON.md`](STATISTICAL_COMPARISON.md) for details.

### To match the paper

Set `NUM_EPISODES = "0:200"` in the configuration cell above. On 3 GPUs, 200 episodes × 19 tasks × 2 models takes many hours — use `tmux` or `screen` to prevent SSH disconnects from killing the process.

### Scaling throughput

We provide several options for simple parallelization. The best settings will depend on your specific hardware setup.

| Flag | Effect |
|------|--------|
| `--num_gpus N` | Use N GPUs, each with its own policy server. |
| `--num_processes N` | Run N parallel episodes *within* each Docker container. Tune by monitoring `nvidia-smi`. |
| `--tasks_per_gpu N` | Run N tasks concurrently on each GPU, each with its own policy server and Docker container. Policy server and simulation share GPU memory, so the ceiling depends on your GPU's VRAM. |


In [ ]:
from tutorials.tutorial_utils import clean_eval_timestamps

# Compare two models. Set NUM_EPISODES="0:200" above to match the paper.
COMPARE_TASKS = "BimanualPutRedBellPepperInBin PutMugOnSaucer TurnCupUpsideDown"

# load_episodes() (run later) rejects anything not under {model}/{Task}/rollouts/{timestamp}/.
# Strip loose timestamp dirs at the OUTPUT_DIR root left by previous tutorial runs.
clean_eval_timestamps(OUTPUT_DIR)

!uv run python vla_foundry/eval/run_evaluation.py TRI-ML/Foundry-Qwen3VLA-2B \
    --num_episodes {NUM_EPISODES} --max_sample_size {MAX_SAMPLE_SIZE} \
    --num_gpus {NUM_GPUS} \
    --tasks {COMPARE_TASKS} \
    --model_name Qwen3VLA-2B-local_evaluation \
    --output_dir {OUTPUT_DIR}

!uv run python vla_foundry/eval/run_evaluation.py TRI-ML/Foundry-VLA-1.5B-full \
    --num_episodes {NUM_EPISODES} --max_sample_size {MAX_SAMPLE_SIZE} \
    --num_gpus {NUM_GPUS} \
    --tasks {COMPARE_TASKS} \
    --model_name Foundry-VLA-1.5B-local_evaluation \
    --output_dir {OUTPUT_DIR}

---

## Step 4: Viewing and Analyzing Results

### Expected output layout

`run_evaluation.py` writes results in this layout:

```
{OUTPUT_DIR}/
├── {model_name}/                          # from --model_name
│   ├── {Task}/                            # e.g. BimanualPutRedBellPepperInBin
│   │   ├── {timestamp}/
│   │   │   └── results.json
│   │   └── .docker.log
```

The dashboard (`load_episodes` / `results_explorer.py`) also accepts a nested variant with an eval-set wrapper — useful when pointing the dashboard directly at `vla_foundry/eval/eval_results/`:

```
{root}/
├── {eval_set}/                            # e.g. OSS
│   ├── {model_name}/
│   │   ├── {Task}/
│   │   │   ├── {timestamp}/
│   │   │   │   └── results.json
```

In the nested variant the dashboard shows models as `{eval_set}/{model_name}`.

### Comparing to paper results

Reference evaluation results from the paper are included in `vla_foundry/eval/eval_results/OSS/`. These are produced by the same open-source Docker simulation image this tutorial uses.

Run the cell below to copy them into your output directory. They will appear alongside your locally reproduced results in the dashboard and statistical tests. The copy function flattens the `OSS/` wrapper so everything lives at `{OUTPUT_DIR}/{model}/`.

> **Non-determinism:** Policy inference is not deterministic — many factors such as reproduction environments can also introduce differences. Locally reproduced results may differ from the paper's numbers.

> **Note:** A `CS/` directory also exists under `eval_results/`; these are results from an internal closed-source simulation and are not reproducible with this tutorial.

In [ ]:
from tutorials.tutorial_utils import copy_foundry_whitepaper_results

copy_foundry_whitepaper_results(OUTPUT_DIR)

In [ ]:
from pathlib import Path

from vla_foundry.eval.data_loading import aggregate_episodes, load_episodes

root = Path(OUTPUT_DIR)
episodes, pending_by, crashed_by, max_sample_size = load_episodes(root)

print(f"Loaded {len(episodes)} episodes")
if max_sample_size:
    print(f"Max sample size per model: {max_sample_size}")
if episodes:
    print(f"\nSample episode:\n{episodes[0]}")

In [ ]:
# Requires: uv sync --group dashboard (for plotly)
from vla_foundry.eval.stats import clopper_pearson_ci

stats = aggregate_episodes(episodes, ci_fn=clopper_pearson_ci, pending_by=pending_by, crashed_by=crashed_by)

# Print a summary table.
print(f"{'Task':<30} {'Model':<15} {'Success Rate':>13} {'N':>5} {'90% CI':>20}")
print("-" * 88)
for s in sorted(stats, key=lambda x: x["pct"], reverse=True):
    ci = f"[{s['ci_low']:.1%}, {s['ci_high']:.1%}]"
    print(f"{s['task']:<30} {s['model']:<15} {s['pct']:>12.1f}% {s['total']:>5} {ci:>20}")

### Viewing Episode Recordings

Each episode loaded by `load_episodes` includes the path to its video recording (if `RECORD_VIDEO=1` was set). The cell below displays the first available recording inline. You can change the filter to view specific tasks, models, or outcomes.

In [ ]:
from IPython.display import Video, display

# Filter episodes (modify to taste):
#   - All episodes: filtered = episodes
#   - Successes only: filtered = [e for e in episodes if e["success"]]
#   - Failures only: filtered = [e for e in episodes if not e["success"]]
#   - Specific task: filtered = [e for e in episodes if e["task"] == "PutMugOnSaucer"]
filtered = episodes

# Display the first episode with a video recording.
for ep in filtered:
    if ep.get("recording_video"):
        print(f"Task: {ep['task']}  Model: {ep['model']}  Episode: {ep['demo_id']}  Success: {ep['success']}")
        print(f"Video: {ep['recording_video']}")
        display(Video(ep["recording_video"], embed=True, width=640))
        break
else:
    print("No recordings found. Set RECORD_VIDEO=1 in the Docker command to enable video recording.")

### Interactive Gradio Dashboard (optional)

For a full interactive experience — summary table, bar/violin/spider charts, paginated episode video playback, and statistical significance testing — launch the Gradio dashboard:

```bash
uv run --group dashboard python vla_foundry/eval/results_explorer.py rollouts/
```

This starts a server at `http://localhost:8505`. Filter by task or model with the dropdowns, and click **Refresh** to pick up new results while evaluation is running.

> **Browser compatibility:** The dashboard has been tested on Chrome. Video playback in the Episode Recordings tab does not work properly on Safari.

To compare multiple models, run evaluations with different `--model_name` values and point the dashboard at the same output directory.

In [ ]:
# Launch the Gradio dashboard (this blocks -- run in a separate terminal or use &).
!uv run --group dashboard python vla_foundry/eval/results_explorer.py {OUTPUT_DIR}

---

## Step 5: Troubleshooting

| Issue | Solution |
|-------|----------|
| **Docker container logs** | Simulation output is saved to `{OUTPUT_DIR}/{model}/{Task}/.docker.log`. Check these if a task fails or hangs. |
| **Connection refused / simulation hangs** | Ensure the policy server is fully initialized (`LBMDiffusionPolicy initialized` in logs) before starting Docker. Check `.policy_server_gpu{N}_slot{M}.log` for errors. |
| **GPU rendering errors (EGL)** | Verify NVIDIA Container Toolkit is installed and `/dev/dri` is accessible. |
| **Permission denied on rollouts** | Run `mkdir -p {OUTPUT_DIR} && chmod 777 {OUTPUT_DIR}` before starting. |
| **Slow first episode** | Normal — Drake downloads model packages and compiles the scene on the first run. |
| **OOM** | Policy server and simulation share the GPU. Reduce `--tasks_per_gpu` or `--num_processes` to lower concurrent memory usage. |
| **Script errors** | If `run_evaluation.py` fails, check the policy server and Docker logs listed in the output. |
| **Stale containers** | `docker kill $(docker ps -q)` to clean up. |
| **Reproducing paper numbers** | Use 200 episodes (`0:200`). Results may vary across machines due to hardware and environment differences. |
| **Diagnosing crashes** | Check `{OUTPUT_DIR}/**/results.json` — episodes with `total_time: 0` and a gRPC traceback in `failure_message` are infrastructure crashes, not eval failures. |

---

## Cleanup

Run this cell to kill all policy servers and Docker evaluation containers.

In [ ]:
# Kill all policy servers on the default port.
!kill $(lsof -ti :{POLICY_PORT}) 2>/dev/null \
    && echo "Killed process(es) on port {POLICY_PORT}." \
    || echo "No process on port {POLICY_PORT}."

# Kill all evaluation Docker containers.
!docker kill $(docker ps -q --filter ancestor={DOCKER_IMAGE}) 2>/dev/null \
    && echo "Docker containers stopped." \
    || echo "No running containers found."